# 02 - 深度学习二分类器（BinaryClassifier）



**方法概述：** 基于FT_cands（文献新发现42颗CN星候选体）作为种子进行数据增强，训练Conv1D/MLP二分类器，在GCS和CNstar独立验证集上评估。



**核心设计：**

1. **数据增强**：对FT_cands种子光谱进行噪声注入、RV偏移、倾斜、mixup、深度调制等增强

2. **Encoder**: Conv1D（3800-5000Å宽波段）或MLP（12维物理特征）

3. **验证集**：GCS（29颗球状星团CN星）+ CNstar（106颗文献CN星）完全独立于训练

4. **Ensemble**：5模型集成，降低方差

In [ ]:
# 导入与数据加载

import sys

from pathlib import Path

_PROJECT_ROOT = Path.cwd()

if str(_PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(_PROJECT_ROOT))



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings('ignore')



print("BinaryClassifier 模块总结")

print("=" * 70)



## 1. 模型架构



### Conv1D Encoder（宽波段 ~1200 pixels, 3800-5000Å）

- Block1: Conv1d(1→32, k=7, s=2) → BN → LeakyReLU → MaxPool(4)

- Block2: Conv1d(32→64, k=5, s=2) → BN → LeakyReLU → MaxPool(4)

- Block3: Conv1d(64→128, k=3) → BN → LeakyReLU

- AdaptiveAvgPool(8) → Linear(1024→256→128→64)



### MLP Encoder（12-D 物理特征）

- Linear(12→64→32→8) + LeakyReLU + Dropout



### 二分类头

- Linear(64→16→1) + Sigmoid

In [ ]:
# 模型架构回顾（从BinaryClassifier/models.py）

import torch

import torch.nn as nn



class SpectralConvEncoder(nn.Module):

    """1D Conv encoder for wide spectral range (3800-5000Å)."""

    def __init__(self, input_dim=1201, latent_dim=64, dropout=0.3):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),

            nn.BatchNorm1d(32), nn.LeakyReLU(0.1), nn.MaxPool1d(4),

            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2, bias=False),

            nn.BatchNorm1d(64), nn.LeakyReLU(0.1), nn.MaxPool1d(4),

            nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),

            nn.BatchNorm1d(128), nn.LeakyReLU(0.1),

        )

        self.pool = nn.AdaptiveAvgPool1d(8)

        conv_out = 128 * 8

        self.head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(conv_out, 256, bias=False), nn.LeakyReLU(0.1), nn.Dropout(dropout),

            nn.Linear(256, 128, bias=False), nn.LeakyReLU(0.1), nn.Dropout(dropout),

            nn.Linear(128, latent_dim, bias=False),

        )

        self.latent_dim = latent_dim



    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.conv(x)

        x = self.pool(x)

        return self.head(x)



# 参数统计

model = SpectralConvEncoder(input_dim=1201, latent_dim=64)

n_params = sum(p.numel() for p in model.parameters())

print(f"Conv1D Encoder 参数量: {n_params:,}")

print(f"  输入: (B, 1201)")

print(f"  输出: (B, 64)")



## 2. 数据增强策略



以FT_cands（42颗）为种子进行增强，每颗生成约37个变体，得到~1500正样本：



| 增强方法 | 数量 | 说明 |

|---------|------|------|

| 噪声注入 | 12 | 高斯噪声 σ=0.005 |

| RV偏移 | 6 | ±30 km/s |

| 光谱倾斜 | 4 | 线性倾斜修正 |

| Mixup | 8 | 正样本间线性混合 |

| 深度调制 | 6 | 吸收线深度调整 |



增强后正样本与负样本1:1平衡训练。

In [ ]:
# 实验配置回顾

config = {

    '数据': {

        '训练种子': 'FT_cands (42颗)',

        '增强后正样本': '~1500',

        '负样本池': '~33,500 (排除已知CN星)',

        '验证集': 'GCS (29颗) + CNstar (106颗)',

        '波长范围': '3800-5000Å (~1201 pixels)',

    },

    '模型': {

        'Encoder': 'Conv1D / MLP (12-D physics)',

        'Latent dim': 64,

        'Dropout': 0.35,

        'Ensemble size': 5,

    },

    '训练': {

        'Epochs': 80,

        'Early stopping patience': 15,

        'Learning rate': 1e-4,

        'Weight decay': 1e-5,

        'Optimizer': 'AdamW',

        'Mixup alpha': 0.2,

    },

}



for section, items in config.items():

    print(f"\n{section}:")

    for k, v in items.items():

        print(f"  {k}: {v}")



## 3. 关键实验结果



实验在BinaryClassifier/目录下进行了系统性的消融和对比实验：



### 主要发现：



**1. 编码器对比（Conv1D vs MLP）**

- Conv1D在GCS验证集上表现更好（更高的recall@k）

- MLP在物理特征空间中更稳定，参数偏差更小

- 宽波段Conv1D能同时捕获CN3839、CN4142、CH4300等多个分子带特征



**2. 增强策略重要性**

- 从42颗种子扩展到~1500正样本是可行的

- Multiplicative增强（噪声+RV+tilt+mixup+depth）效果最佳

- 仅用FT_cands增强即可，不需要GCS参与训练



**3. 验证集独立性**

- GCS和CNstar完全独立于FT_cands训练集

- 模型泛化能力经过严格检验



**4. 与后续ML方法的关系**

- BinaryClassifier验证了深度学习在CN星检测上的可行性

- 但FT_cands增强的泛化能力有限

- 后续转向XGBoost + PU Bagging（Notebook 03）和AE特征学习（Notebook 04）

In [ ]:
# 实验结果总结

print("=" * 70)

print("BinaryClassifier 核心实验结论")

print("=" * 70)



conclusions = [

    "1. FT_cands增强可生成足够正样本训练二分类器（42→~1500）",

    "2. Conv1D(宽波段)在GCS上recall优于MLP(物理特征)",

    "3. 5模型Ensemble显著降低预测方差，提高候选体可靠性",

    "4. 验证集(GCS+CNstar)完全独立于训练，检验了泛化能力",

    "5. 限制：FT_cands增强的多样性受限于种子本身，难覆盖所有CN星子类",

    "6. 启示：需要更系统的方法 — 转向PU Learning + 更丰富的特征表示",

]

for c in conclusions:

    print(c)



print("\n" + "=" * 70)

print("BinaryClassifier → ML/XGBoost PU → SpectraAE 的技术演进路线：")

print("=" * 70)

print("阶段1: BinaryClassifier — 验证DL可行性，但增强有限")

print("阶段2: ML/XGB_PU — PU Bagging无需增强，直接利用全数据集")

print("阶段3: SpectraAE — AE学习光谱表示，PU在特征空间更高效")



## 4. 模型checkpoint与产出



训练好的模型保存在 `BinaryClassifier/checkpoints/`：

- Conv1D ensemble（5个模型）

- MLP ensemble（5个模型）



候选体预测保存在 `BinaryClassifier/candidates_ft_conv1d.csv` 和 `candidates_ft_mlp.csv`。



**后续改进方向：**

1. 尝试更多样的增强策略（GAN、VAE等）

2. 引入自监督预训练

3. 多任务学习（同时预测stellar parameters和CN概率）